In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
full_flight_data = spark.table("workspace.silver.full_flight_data")

### Delay Analysis 

In [0]:
# Over all delay metrics analysis

overall_metrics = full_flight_data \
        .filter(F.col("cancelled") == 0) \
        .agg(
            F.avg("arr_delay_mins").alias("average_arrival_delay"),
            F.avg("dep_delay_mins").alias("average_departure_delay"),
            (F.sum(F.when(F.col("ArrDel15") > 0, 1).otherwise(0)) / F.count("*") * 100).alias("percentage_arrivals_delayed_15min"),
            (F.sum(F.when(F.col("DepDel15") > 0, 1).otherwise(0)) / F.count("*") * 100).alias("percentage_departure_delayed_15min")
        )
    

In [0]:
overall_metrics.display()

###  Delay by Carrier

In [0]:
delay_by_carrier = full_flight_data \
    .filter(F.col("cancelled") == 0) \
    .groupBy("carrier_name") \
    .agg(
        F.avg("arr_delay_mins").alias("avg_arrival_delay"),
        (F.sum(F.when(F.col("ArrDel15") > 0, 1).otherwise(0)) / F.count("*") * 100).alias("percentage_arrivals_delayed"),
        F.count("*").alias("total_flights")
    ).filter(F.col("carrier_name").isNotNull()).orderBy(F.desc("avg_arrival_delay"))

In [0]:
delay_by_carrier.orderBy(F.col('avg_arrival_delay').desc(), F.col('percentage_arrivals_delayed').desc()).display()


### Delay by Origin Airport


In [0]:
delay_by_origin_airport = full_flight_data \
  .filter(F.col("cancelled") == 0) \
  .groupBy("origin_airport_name") \
  .agg(
      F.avg("dep_delay_mins").alias("avg_departure_delay"),
      (F.sum(F.when(F.col("DepDel15") > 0, 1).otherwise(0)) / F.count("*") * 100).alias("percentage_departures_delayed"),
      F.count("*").alias("total_departing_flights")
  ).filter(F.col("origin_airport_name").isNotNull()).orderBy(F.desc("avg_departure_delay"))

In [0]:
delay_by_origin_airport.display()

### Delay by Route

In [0]:
delay_by_route = full_flight_data \
        .filter(F.col("cancelled") == 0) \
        .groupBy("origin_airport_name", "dest_airport_name") \
        .agg(F.avg("arr_delay_mins").alias("avg_arrival_delay"), F.count("*").alias("total_flights")) \
        .filter(F.col("origin_airport_name").isNotNull() & F.col("dest_airport_name").isNotNull()) \
        .orderBy(F.desc("avg_arrival_delay"))

In [0]:
delay_by_route.display()

### Delay Cause Analysis

In [0]:
delay_cause_summary = full_flight_data \
  .filter((F.col("cancelled") == 0) & (F.col("arr_delay_mins") > 0)) \
  .agg(
      F.avg("carrier_delay").alias("avg_carrier_delay"), F.sum("carrier_delay").alias("total_carrier_delay"),
      F.avg("weather_delay").alias("avg_weather_delay"), F.sum("weather_delay").alias("total_weather_delay"),
      F.avg("nas_delay").alias("avg_nas_delay"), F.sum("nas_delay").alias("total_nas_delay"),
      F.avg("security_delay").alias("avg_security_delay"), F.sum("security_delay").alias("total_security_delay"),
      F.avg("late_aircraft_delay").alias("avg_late_aircraft_delay"), F.sum("late_aircraft_delay").alias("total_late_aircraft_delay")
  )

delay_cause_summary = delay_cause_summary.withColumn("total_delay_sum", F.col("total_carrier_delay") + F.col("total_weather_delay") + F.col("total_nas_delay") + F.col("total_security_delay") + F.col("total_late_aircraft_delay"))

delay_cause_summary = delay_cause_summary.select("*",
    (F.when(F.col("total_delay_sum") > 0, (F.col("total_carrier_delay") / F.col("total_delay_sum")) * 100).otherwise(0)).alias("percent_carrier"),
    (F.when(F.col("total_delay_sum") > 0, (F.col("total_weather_delay") / F.col("total_delay_sum")) * 100).otherwise(0)).alias("percent_weather"),
    (F.when(F.col("total_delay_sum") > 0, (F.col("total_nas_delay") / F.col("total_delay_sum")) * 100).otherwise(0)).alias("percent_nas"),
    (F.when(F.col("total_delay_sum") > 0, (F.col("total_security_delay") / F.col("total_delay_sum")) * 100).otherwise(0)).alias("percent_security"),
    (F.when(F.col("total_delay_sum") > 0, (F.col("total_late_aircraft_delay") / F.col("total_delay_sum")) * 100).otherwise(0)).alias("percent_late_aircraft")
)

delay_cause_summary.display()

### Cancellation Reason Analysis


In [0]:
cancel_code_mapping = spark.createDataFrame([
        ('A', 'Carrier'), ('B', 'Weather'), ('C', 'NAS'), ('D', 'Security')
    ], ["CancellationCode", "ReasonDescription"])

cancellation_reasons = full_flight_data \
    .filter(F.col("cancelled") == 1) \
    .groupBy("CancellationCode") \
    .agg(F.count("*").alias("cancellation_count")) \
    .join(cancel_code_mapping, "CancellationCode", "left") \
    .orderBy(F.desc("cancellation_count"))
cancellation_reasons.display()

### Cancellation Rate by Carrier

In [0]:
cancellation_rate_carrier = full_flight_data \
        .groupBy("carrier_name") \
        .agg((F.sum("cancelled") / F.count("*") * 100).alias("cancellation_rate_percent")) \
        .filter(F.col("carrier_name").isNotNull()) \
        .orderBy(F.desc("cancellation_rate_percent"))

cancellation_rate_carrier.display()

### Cancellation Rate by Route

In [0]:
cancellation_rate_route = (full_flight_data 
  .groupBy("origin_airport_name", "dest_airport_name") 
  .agg((F.sum("cancelled") / F.count("*") * 100).alias("cancellation_rate_percent"), F.count("*").alias("total_flights")) 
  .filter(F.col("origin_airport_name").isNotNull() & F.col("dest_airport_name").isNotNull()) 
  .filter(F.col("total_flights") > 50) # Optional: Filter for routes with significant traffic
  .orderBy(F.desc("cancellation_rate_percent")))
        
cancellation_rate_route.display()


### Operational Efficiency Analysis

In [0]:
valid_efficiency_flights = full_flight_data \
        .filter((F.col("cancelled") == 0) & (F.col("diverted") == 0)) \
        .filter(F.col("taxi_out").isNotNull() & F.col("taxi_in").isNotNull() & \
                F.col("air_time").isNotNull() & F.col("actual_elapsed_time").isNotNull() & \
                F.col("crs_elapsed_time").isNotNull())

### Taxi Time Analysis

In [0]:
avg_taxi_out = valid_efficiency_flights \
    .groupBy("origin_airport_name") \
    .agg(F.avg("taxi_out").alias("avg_taxi_out_time")) \
    .filter(F.col("origin_airport_name").isNotNull()) \
    .orderBy(F.desc("avg_taxi_out_time"))

avg_taxi_out.display()

avg_taxi_in = valid_efficiency_flights \
    .groupBy("dest_airport_name") \
    .agg(F.avg("taxi_in").alias("avg_taxi_in_time")) \
    .filter(F.col("dest_airport_name").isNotNull()) \
    .orderBy(F.desc("avg_taxi_in_time"))
avg_taxi_in.display()

In [0]:
efficiency = valid_efficiency_flights \
        .groupBy("carrier_name", "origin_airport_name", "dest_airport_name") \
        .agg(
            F.avg("air_time").alias("avg_air_time_mins"),
            F.count("*").alias("total_flights")
         ) \
        .filter(F.col("total_flights") > 30) \
        .orderBy(F.desc("avg_air_time_mins"))

efficiency.display()

In [0]:
ground_time_analysis = valid_efficiency_flights \
        .withColumn("ground_time", F.col("actual_elapsed_time") - F.col("air_time")) \
        .groupBy("carrier_name", "origin_airport_name", "dest_airport_name") \
        .agg(F.avg("ground_time").alias("avg_ground_time_mins"), F.count("*").alias("total_flights")) \
        .filter(F.col("total_flights") > 30) \
        .orderBy(F.desc("avg_ground_time_mins"))
ground_time_analysis.display()

### Bring all together in a single table

In [0]:
key_cols = ["carrier_name", "origin_airport_name", "dest_airport_name"]

# Delay by route
delay_by_route = full_flight_data.filter(F.col("cancelled") == 0) \
    .groupBy(*key_cols) \
    .agg(
        F.avg("arr_delay_mins").alias("avg_arrival_delay"),
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("ArrDel15") > 0, 1).otherwise(0)).alias("num_arr_delayed")
    )

# Cancellation by route
cancellation_rate_route = full_flight_data.groupBy(*key_cols) \
    .agg(
        F.sum("cancelled").alias("cancelled_flights"),
        F.count("*").alias("total_flights_cancel")
    ).withColumn("cancellation_rate", F.col("cancelled_flights") / F.col("total_flights_cancel") * 100)

# Efficiency - padding
efficiency_padding = full_flight_data.filter((F.col("cancelled") == 0) & (F.col("diverted") == 0)) \
    .withColumn("flight_time_padding", F.col("actual_elapsed_time") - F.col("crs_elapsed_time")) \
    .groupBy(*key_cols) \
    .agg(
        F.avg("flight_time_padding").alias("avg_padding_mins"),
        F.avg("air_time").alias("avg_air_time_mins")
    )

# Efficiency - ground time
ground_time = full_flight_data.filter((F.col("cancelled") == 0) & (F.col("diverted") == 0)) \
    .withColumn("ground_time", F.col("actual_elapsed_time") - F.col("air_time")) \
    .groupBy(*key_cols) \
    .agg(F.avg("ground_time").alias("avg_ground_time"))

# Join all metrics
final_dashboard_table = delay_by_route \
    .join(cancellation_rate_route, key_cols, "outer") \
    .join(efficiency_padding, key_cols, "outer") \
    .join(ground_time, key_cols, "outer")

In [0]:
%sql
Create schema if not exists workspace.gold

In [0]:
final_dashboard_table.write.mode("overwrite").saveAsTable("workspace.gold.final_dashboard_table")

In [0]:
%sql
select * from workspace.gold.final_dashboard_table limit 10;